# SEO 文章生成流水线 · Colab 版

把线上那条七步流水线搬到 Colab 跑，**用你自己的 API Key**。

流程与线上完全一致（直接复用仓库代码，不是重写）：

| 步 | 干什么 |
|---|---|
| ① | 全网搜索主/次关键词的现有内容 |
| ② | 抓 Reddit 真人讨论（找竞品没覆盖的痛点） |
| ③ | 抽取**带出处的事实清单**——正文只准用清单里的数字 |
| ④ | 判断主题类型 → 生成 EEAT 大纲 |
| ⑤ | **你审批 / 改大纲**（独立单元格，不通过就不往下走） |
| ⑥ | 写正文 + SEO 元数据 + 交付前后处理 |
| ⑦ | 润色到目标阅读年级（带闸门：够好就不整篇重写） |

**与线上唯一的区别：搜索侧用 Tavily。** 它的 `/search` 直接返回正文，搜索和抓取一步完成；
Reddit 找帖也走它。

### 需要两个 Key

| Key | 去哪拿 | 用途 |
|---|---|---|
| `GEMINI_API_KEY` | aistudio.google.com/apikey | 写作 |
| `TAVILY_API_KEY` | tavily.com | 搜索 + 找 Reddit 帖 |

填进 Colab 左侧 🔑 **密钥（Secrets）**，名字照上面写，打开「笔记本访问权限」。

## 1 · 装依赖、拉代码

In [ ]:
#@title 安装依赖并克隆仓库 { display-mode: "form" }
!pip -q install httpx pydantic pydantic-settings python-docx nest_asyncio 2>/dev/null

import os, shutil, subprocess, sys

REPO = "/content/pagezenith"
if os.path.isdir(REPO):
    shutil.rmtree(REPO)          # 每次重拉，保证跟线上同步
subprocess.run(["git", "clone", "--depth", "1", "-q",
                "https://github.com/hitechhamster/pagezenith.git", REPO], check=True)

sys.path.insert(0, REPO + "/api")   # 仓库里所有 import 以 api/ 为根

import nest_asyncio
nest_asyncio.apply()   # Colab 自带事件循环，不打这个补丁跑不了 async

ver = subprocess.run(["git", "-C", REPO, "log", "-1", "--format=%h %s"],
                     capture_output=True, text=True).stdout.strip()
print("仓库版本：", ver)

## 2 · 填 Key

In [ ]:
#@title 读取密钥 { display-mode: "form" }
import os

def _get(name):
    # 优先 Colab 密钥；没配就当场问 —— 不把 key 写进 notebook
    try:
        from google.colab import userdata
        v = userdata.get(name)
        if v:
            return v.strip()
    except Exception:
        pass
    import getpass
    return getpass.getpass(name + ": ").strip()

os.environ["GEMINI_API_KEY"] = _get("GEMINI_API_KEY")
os.environ["TAVILY_API_KEY"] = _get("TAVILY_API_KEY")
print("GEMINI", "OK" if os.environ["GEMINI_API_KEY"] else "缺失",
      "| TAVILY", "OK" if os.environ["TAVILY_API_KEY"] else "缺失")

## 3 · Tavily 适配层

仓库里 `search()` 本来就带 tavily 分支，配上 key 就能用。

**要自己补的只有 Reddit 那一处**：`RedditClient` 是从搜索结果里的 reddit 链接反查帖子 ID 的，
它期望一个带 `.fetch_serp()` 的对象。下面给它一个 Tavily 版替身——用 `include_domains`
把搜索直接限定在 reddit.com，比线上「搜 `<关键词> reddit` 再按 URL 过滤」更准。

In [ ]:
#@title Tavily 适配层 { display-mode: "form" }
import httpx
from dataclasses import dataclass
from tools.seo_gap.config import Settings
from tools.seo_gap.clients import reddit as reddit_mod


@dataclass
class _Item:
    # RedditClient 只用到 .url 和 .title
    url: str
    title: str
    rank: int = 0


class TavilySerp:
    # 给 RedditClient 用的 SERP 替身：Tavily 搜索限定在 reddit.com
    def __init__(self, s):
        self.s = s

    async def fetch_serp(self, keyword, location_code=2840, language_code="en", depth=20):
        q = keyword[:-7].strip() if keyword.lower().endswith(" reddit") else keyword
        async with httpx.AsyncClient(timeout=self.s.tavily_timeout, trust_env=False) as c:
            r = await c.post(
                self.s.tavily_base_url + "/search",
                headers={"Authorization": "Bearer " + self.s.tavily_key},
                json={"query": q, "search_depth": "advanced",
                      "include_domains": ["reddit.com"], "max_results": min(depth, 20)},
            )
            r.raise_for_status()
            data = r.json()
        return [_Item(url=it.get("url", ""), title=it.get("title", ""), rank=i)
                for i, it in enumerate(data.get("results", []), 1)]

    async def fetch_serp_full(self, keyword, location_code=2840, language_code="en", depth=10):
        items = await self.fetch_serp(keyword, location_code, language_code, depth)
        return {"items": items, "paa": [], "related": []}

    async def fetch_autocomplete(self, keyword, language_code="en"):
        return []


# 接管 RedditClient 的 SERP 依赖（它在 __init__ 里按 serp_provider 二选一）
_orig_init = reddit_mod.RedditClient.__init__

def _patched_init(self, settings=None):
    _orig_init(self, settings)
    self.serp = TavilySerp(self.s)

reddit_mod.RedditClient.__init__ = _patched_init
print("Reddit 找帖已切到 Tavily")

## 4 · 配置

In [ ]:
#@title 运行参数 { display-mode: "form" }
import os
from tools.seo_gap.config import Settings
from tools.seo_writer.providers import LLM, resolve_llm
from tools.seo_writer.workflow import SEOWriter

S = Settings(
    gemini_api_key=os.environ["GEMINI_API_KEY"],
    tavily_key=os.environ["TAVILY_API_KEY"],
    serp_provider="tavily",
    outbound_proxy="",        # Colab 在墙外，直连即可（线上那台香港机才要走隧道）
    use_mocks=False,
)

TIER = "pro"     # 模型按任务分工，映射在仓库的 billing/pricing.py 里

USAGE = {"tin": 0, "tout": 0}

def _note(model, tin, tout):
    USAGE["tin"] += tin or 0
    USAGE["tout"] += tout or 0

def new_writer():
    # resolve_llm 会按档位把「大纲/正文/润色/杂活」分别映射到各自的模型
    # （映射表在仓库 billing/pricing.py），和线上完全一致
    target = resolve_llm(S, TIER)
    return SEOWriter(S, LLM(target, S, usage_sink=_note), "tavily")

import asyncio

def run(coro):
    # Colab 里已有事件循环（第 1 格打过 nest_asyncio 补丁），用 run_until_complete；
    # 普通 python 里没有循环，用 asyncio.run。两种环境都能跑。
    try:
        loop = asyncio.get_running_loop()
    except RuntimeError:
        return asyncio.run(coro)
    return loop.run_until_complete(coro)

print("配置完成 · 档位", TIER)

## 5 · 第①–④步：搜索 → Reddit → 事实清单 → 大纲

改下面的参数再运行。**跑完大纲会打印出来，先别急着往下走。**

In [ ]:
#@title 生成大纲 { display-mode: "form" }
MAIN_KEYWORD      = "best shopify apps for abandoned cart recovery"  #@param {type:"string"}
SECONDARY_KEYWORD = "abandoned cart recovery app shopify"            #@param {type:"string"}
TOPIC             = "listicle comparing abandoned cart recovery apps for Shopify"  #@param {type:"string"}
WORDCOUNT         = 1500      #@param {type:"integer"}
LANGUAGE          = "English" #@param ["English","Chinese (Simplified)","Chinese (Traditional)","Spanish","French","German","Japanese","Portuguese","Korean","Italian","Indonesian"]
#@markdown 「具体要求」优先级最高：写榜单 / 指定读者 / 必答问题 / 推荐产品都写这里
SPECIFIC = "Write as a ranked listicle of 6-8 apps. For each: what it does best, pricing, who it suits, one honest limitation. Include a comparison table near the top."  #@param {type:"string"}

import asyncio, re

wf = new_writer()
CTX = {
    "main_keyword": MAIN_KEYWORD.strip(), "secondary_keyword": SECONDARY_KEYWORD.strip(),
    "topic": TOPIC.strip(), "specific": SPECIFIC, "language": LANGUAGE,
    "wordcounts": WORDCOUNT, "tier": TIER, "revise_count": 0,
    "enable_images": False, "images_per_article": 0, "image_style": "auto", "voice": "",
}

async def build_outline(ctx):
    print("① 全网搜索…", flush=True)
    ctx["main_search"], ctx["sec_search"] = await wf.search_context(
        ctx["main_keyword"], ctx["secondary_keyword"])

    print("② 抓 Reddit 真实讨论…", flush=True)
    ctx["reddit_context"] = await wf.reddit_context(ctx["main_keyword"])
    threads = re.findall(r"^\[r/([^\]]+)\]\s*(.+?)（(\d+)赞/(\d+)评）\s*$",
                         ctx["reddit_context"] or "", re.M)
    print("   拿到 %d 条讨论" % len(threads))
    for sub, title, score, ncom in threads[:5]:
        print("     r/%s · %s赞/%s评 · %s" % (sub, score, ncom, title[:60]))

    print("③ 抽取可核实事实（带出处）…", flush=True)
    ctx["facts"] = await wf.extract_facts(ctx)
    facts = [l for l in (ctx["facts"] or "").splitlines() if " — " in l or " - " in l]
    print("   锁定 %d 条事实 —— 正文只准用清单里的数字" % len(facts))
    for f in facts[:5]:
        print("    ", f.strip()[:100])

    print("④ 判断主题类型 → 写大纲…", flush=True)
    ctx["topic_type"] = await wf.classify_topic_type(ctx["main_keyword"], ctx["topic"])
    print("   主题类型：", ctx["topic_type"])
    buf = []
    async for piece in wf.stream_outline(ctx):
        buf.append(piece)
    ctx["outline"] = "".join(buf)
    return ctx

CTX = run(build_outline(CTX))
print("\n" + "=" * 70 + "\n大纲：\n" + "=" * 70)
print(CTX["outline"])

## 6 · 第⑤步：审批大纲

这是这条流水线和「一键生成」最大的区别——**不通过就不往下写**。三种走法：

- **直接通过**：跳过这格，往下跑第 7 节
- **让 AI 改**：填 `FEEDBACK` 后运行本格（可反复跑）
- **自己改**：直接给 `CTX["outline"]` 赋新值，再往下跑

In [ ]:
#@title 让 AI 改大纲（可反复运行） { display-mode: "form" }
FEEDBACK = "把价格对比表提到最前面；每个 app 补一句「不适合谁」"  #@param {type:"string"}

import asyncio

old_outline = CTX["outline"]
_buf = []

async def _revise():
    async for piece in wf.stream_revise(CTX, FEEDBACK):
        _buf.append(piece)

run(_revise())
CTX["outline"] = "".join(_buf)
CTX["revise_count"] = CTX.get("revise_count", 0) + 1

print("改了什么：", wf.outline_section_diff(old_outline, CTX["outline"]))
print("=" * 70)
print(CTX["outline"])

## 7 · 第⑥步：写正文 + SEO 元数据 + 交付前后处理

In [ ]:
#@title 写正文 { display-mode: "form" }
import asyncio
from tools.seo_writer import prose_audit
from tools.seo_writer.postfix import postfix
from tools.seo_writer.workflow import reading_grade

async def write_article(ctx):
    print("⑥ 撰写正文…（长文 1–3 分钟）", flush=True)
    buf = []
    async for piece in wf.stream_article(ctx):
        buf.append(piece)
    text = "".join(buf)

    cjk = ctx["language"].startswith("Chinese")
    words = len(text) if cjk else len(text.split())
    print("   %d %s（目标 %d，%.0f%%）" % (words, "字" if cjk else "词",
                                        ctx["wordcounts"], words / ctx["wordcounts"] * 100))
    print("   阅读年级 FK %.1f" % reading_grade(text, ctx["language"]))

    print("   生成 SEO 标题与描述…", flush=True)
    seo = await wf.generate_seo(text, ctx["main_keyword"], ctx["language"])

    print("   交付前后处理（假经验句 / 塞词）…", flush=True)
    kws = [ctx["main_keyword"], ctx["secondary_keyword"]]
    text, fixes = await postfix(text, kws, ctx.get("facts", ""), wf.llm.complete)
    for f in fixes:
        print("     修正：", f[:90])

    # 免费的确定性体检：不改文章，只告诉你哪里还有 AI 指纹
    rep = prose_audit.audit(text)
    c = rep["counts"]
    print("   体检：P0 %d · P1 %d · P2 %d" % (c["P0"], c["P1"], c["P2"]))
    for f in rep["findings"]:
        print("     [%s] %s：%s" % (f["severity"], f["rule"], f["detail"]))

    unlisted = prose_audit.unlisted_numbers(text, ctx.get("facts", ""))
    if unlisted:
        print("   ⚠️ 清单外的数字 %d 处，请自己核实：" % len(unlisted), unlisted[:5])
    return text, seo

ARTICLE, SEO = run(write_article(CTX))
print("\n" + "=" * 70)
print("SEO 标题：", SEO.get("title", ""))
print("SEO 描述：", SEO.get("description", ""))

## 8 · 第⑦步：润色（带闸门——够好就不整篇重写）

In [ ]:
#@title 润色 { display-mode: "form" }
import asyncio
from tools.seo_writer.workflow import reading_grade

async def do_polish(text):
    # 闸门在 stream_polish 内部自动跑：FK>12 走整篇重写(full)，已达标只做轻润色(light)。
    # 这里单独调 polish_mode 只为把判断打印出来 —— ⚠️ 不能把 mode 当参数传进去，
    # stream_polish 的第三个位置参数是 strict（结构重试用），传字符串会让它每次都走重试模板。
    mode, why = wf.polish_mode(text, CTX["language"])
    print("⑦ 力度 %s（%s）" % (mode, why))
    buf = []
    async for piece in wf.stream_polish(CTX, text):
        buf.append(piece)
    out = "".join(buf)

    # 两道硬闸门：润色不许新增信息，也不许把结构改坏
    added = wf.polish_added_numbers(text, out)
    if added:
        print("   ✗ 润色新增了原文没有的数字（%s）→ 回退" % "、".join(added[:4]))
        return text
    broke = wf.polish_broke_structure(text, out)
    if broke:
        print("   ✗ 润色破坏了结构（%s）→ 回退" % broke)
        return text
    print("   FK %.1f → %.1f" % (reading_grade(text, CTX["language"]),
                                 reading_grade(out, CTX["language"])))
    return out

FINAL = run(do_polish(ARTICLE))
print("\n完成。用量：输入 %s / 输出 %s tokens" % (format(USAGE["tin"], ","), format(USAGE["tout"], ",")))

## 9 · 导出 Word / Markdown

In [ ]:
#@title 导出并下载 { display-mode: "form" }
import pathlib
from tools.seo_writer.docx_export import build_docx, sanitize_filename

name = sanitize_filename(SEO.get("title") or CTX["main_keyword"])[:60]
out = pathlib.Path("/content/out")
out.mkdir(exist_ok=True)

docx_path = out / (name + ".docx")
docx_path.write_bytes(build_docx(FINAL))

md_path = out / (name + ".md")
md_path.write_text(
    "<!-- SEO Title: %s -->\n<!-- SEO Description: %s -->\n\n%s"
    % (SEO.get("title", ""), SEO.get("description", ""), FINAL),
    encoding="utf-8")

print("已导出：", docx_path.name, "/", md_path.name)
try:
    from google.colab import files
    files.download(str(docx_path))
except Exception:
    pass

## 10 · 批量模式

一次跑一批关键词。**落盘 + 断点续跑**：跑过的自动跳过，中途断了重跑不会重复花钱。

批量模式下大纲审批自动通过——批量的前提就是不逐篇看。要保质量用上面的单篇流程。

In [ ]:
#@title 批量生成 { display-mode: "form" }
#@markdown 一行一个：`主关键词｜次关键词｜主题`（后两项可省）
KEYWORDS = "best shopify apps for abandoned cart recovery｜abandoned cart recovery app shopify｜listicle\nshopify email marketing automation｜klaviyo alternatives｜how-to"  #@param {type:"string"}
BATCH_WORDCOUNT = 1500  #@param {type:"integer"}
BATCH_LANGUAGE  = "English"  #@param ["English","Chinese (Simplified)","Chinese (Traditional)","Spanish","French","German","Japanese","Portuguese","Korean","Italian","Indonesian"]
BATCH_SPECIFIC  = ""  #@param {type:"string"}

import asyncio, json, pathlib, time, traceback
from tools.seo_writer.docx_export import build_docx, sanitize_filename
from tools.seo_writer.postfix import postfix

OUT = pathlib.Path("/content/out")
OUT.mkdir(exist_ok=True)
STATE = OUT / "_done.json"
done = json.loads(STATE.read_text(encoding="utf-8")) if STATE.exists() else {}

rows = [l.strip() for l in KEYWORDS.replace("\\n", "\n").strip().splitlines() if l.strip()]
print("共 %d 条，已完成 %d 条\n" % (len(rows), len(done)))

async def run_one(line):
    parts = [p.strip() for p in line.replace("|", "｜").split("｜")]
    main = parts[0]
    sec = parts[1] if len(parts) > 1 and parts[1] else main
    topic = parts[2] if len(parts) > 2 and parts[2] else "informational article"

    w = new_writer()
    ctx = {"main_keyword": main, "secondary_keyword": sec, "topic": topic,
           "specific": BATCH_SPECIFIC, "language": BATCH_LANGUAGE,
           "wordcounts": BATCH_WORDCOUNT, "tier": TIER, "revise_count": 0,
           "enable_images": False, "images_per_article": 0,
           "image_style": "auto", "voice": ""}

    ctx["main_search"], ctx["sec_search"] = await w.search_context(main, sec)
    ctx["reddit_context"] = await w.reddit_context(main)
    ctx["facts"] = await w.extract_facts(ctx)
    ctx["topic_type"] = await w.classify_topic_type(main, topic)

    buf = []
    async for p in w.stream_outline(ctx):
        buf.append(p)
    ctx["outline"] = "".join(buf)

    buf = []
    async for p in w.stream_article(ctx):
        buf.append(p)
    text = "".join(buf)

    seo = await w.generate_seo(text, main, BATCH_LANGUAGE)
    text, _ = await postfix(text, [main, sec], ctx.get("facts", ""), w.llm.complete)

    buf = []
    async for p in w.stream_polish(ctx, text):   # 力度闸门在里面自动判
        buf.append(p)
    out = "".join(buf)
    if not w.polish_added_numbers(text, out) and not w.polish_broke_structure(text, out):
        text = out
    return text, seo

for i, line in enumerate(rows, 1):
    key = line.replace("|", "｜").split("｜")[0].strip()
    if key in done:
        print("[%d/%d] 跳过（已完成）：%s" % (i, len(rows), key))
        continue
    t0 = time.time()
    print("[%d/%d] %s …" % (i, len(rows), key), flush=True)
    try:
        text, seo = run(run_one(line))
        name = sanitize_filename(seo.get("title") or key)[:60]
        (OUT / (name + ".docx")).write_bytes(build_docx(text))
        (OUT / (name + ".md")).write_text(
            "<!-- SEO Title: %s -->\n<!-- SEO Description: %s -->\n\n%s"
            % (seo.get("title", ""), seo.get("description", ""), text), encoding="utf-8")
        done[key] = name
        STATE.write_text(json.dumps(done, ensure_ascii=False, indent=1), encoding="utf-8")
        print("      完成 %s（%.0fs）" % (name, time.time() - t0))
    except Exception as exc:
        print("      失败：", exc)
        traceback.print_exc(limit=1)

print("\n全部结束。用量合计：输入 %s / 输出 %s tokens"
      % (format(USAGE["tin"], ","), format(USAGE["tout"], ",")))

In [ ]:
#@title 打包下载全部结果 { display-mode: "form" }
!cd /content && zip -qr articles.zip out -x "out/_done.json"
try:
    from google.colab import files
    files.download("/content/articles.zip")
except Exception:
    print("非 Colab 环境，文件在 /content/articles.zip")

---

### 排查

| 现象 | 原因 / 怎么办 |
|---|---|
| `缺少 Tavily API Key` | 第 2 格没读到密钥，或 Colab 密钥没开「笔记本访问权限」 |
| Reddit 那步 0 条 | 该关键词确实没人讨论；不阻断流程，文章照写 |
| 正文返回空或被截断 | 降低目标字数重试；仓库对思考型模型已有 `reasoning_effort` 压制 |
| 想跟线上完全一致 | 删掉第 3 格的适配层，配 `SERPER_KEY` 并把 `serp_provider` 设回 `serper` |

代码全部来自 `github.com/hitechhamster/pagezenith`，第 1 格每次重拉，线上改了这里自动同步。